In [1]:
import pandas as pd
import numpy as np

# Load datasets
patients = pd.read_csv("patients.csv")
labs = pd.read_csv("labs.csv")
outcomes = pd.read_csv("outcomes.csv")
diagnoses = pd.read_csv("diagnoses.csv")

# Merge datasets
patients = patients.merge(diagnoses, on="DiagnosisID")
patients = patients.merge(outcomes, on="OutcomeID")

# Convert dates
patients["AdmissionDate"] = pd.to_datetime(patients["AdmissionDate"])
patients["DischargeDate"] = pd.to_datetime(patients["DischargeDate"])

# Length of stay
patients["LengthOfStay"] = (
    patients["DischargeDate"] -
    patients["AdmissionDate"]
).dt.days

# Encode Outcome
patients["OutcomeEncoded"] = patients["OutcomeName"].map({
    "Recovered":0,
    "Complicated":1,
    "Diseased":1
})

# Count abnormal labs
abnormal_conditions = {
    "Blood Sugar": lambda x: x > 120,
    "Cholestrol": lambda x: x > 200,
    "Heamoglobin": lambda x: x < 13
}

def count_abnormal(patient_id):
    patient_lab = labs[labs["PatientID"] == patient_id]

    count = 0

    for test, condition in abnormal_conditions.items():
        result = patient_lab[patient_lab["TestName"] == test]
        count += result["Result"].apply(condition).sum()

    return count

patients["AbnormalCounts"] = patients["PatientID"].apply(count_abnormal)

# Features
features = patients[
    ["Age",
     "LengthOfStay",
     "TreatmentCost",
     "AbnormalCounts"]
]

target = patients["OutcomeEncoded"]

# Remove missing values
mask = ~target.isna()

features = features[mask]
target = target[mask]

# Train-test split
from sklearn.model_selection import train_test_split

x_train, x_test, y_train, y_test = train_test_split(
    features,
    target,
    test_size=0.2,
    random_state=42,
    stratify=target
)

# Scale features
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()

x_train = scaler.fit_transform(x_train)
x_test = scaler.transform(x_test)

# Logistic Regression
from sklearn.linear_model import LogisticRegression

model = LogisticRegression(
    max_iter=1000,
    class_weight="balanced"
)

model.fit(x_train, y_train)

# Prediction
y_pred = model.predict(x_test)

from sklearn.metrics import accuracy_score, classification_report

print("Accuracy:", accuracy_score(y_test, y_pred))
print(classification_report(y_test, y_pred))

# Save scaler and model
import joblib

joblib.dump(model, "risk_model.pkl")
joblib.dump(scaler, "scaler.pkl")

print("Model Saved")

Accuracy: 0.48148148148148145
              precision    recall  f1-score   support

         0.0       0.47      0.62      0.53        13
         1.0       0.50      0.36      0.42        14

    accuracy                           0.48        27
   macro avg       0.49      0.49      0.47        27
weighted avg       0.49      0.48      0.47        27

Model Saved
